# Metasmith Starter Notebook
Welcome! This is a starter notebook provided for you to interact with Metasmith.
If you get stuck, you can read [the docs](https://metasmith.readthedocs.io/en/latest/), or check out [the repository on GitHub](https://github.com/hallamlab/Metasmith)!

To begin, import the tools and structures needed from the Metasmith python API. We will also load some resources from the standard library using `Std()`.

In [ ]:
import os
import ipynbname
from IPython.display import SVG, display
from pathlib import Path
from metasmith.python_api import Agent, ContainerRuntime
from metasmith.python_api import DataTypeLibrary, DataInstanceLibrary, TransformInstanceLibrary
from metasmith.python_api import Source, SshSource
from metasmith.python_api import Resources, Size, Duration

In [ ]:
agent_home = SshSource(host="host", path=Path("/path/on/host")).AsSource()
smith = Agent(
    home = agent_home,
    runtime=ContainerRuntime.APPTAINER,
    setup_commands=[
        '...'
    ]
)

smith.Deploy()

In [ ]:
os.system(f"[ -e epi300.gbk ] || wget -q https://github.com/hallamlab/MetasmithLibraries/releases/download/data.epi300.1/epi300.gbk")

In [ ]:
notebook_name = ipynbname.name()
MLIB = Path("MetasmithLibraries")
CACHE = Path("./cache")
in_dir = CACHE/f"{notebook_name}/inputs.xgdb"

inputs = DataInstanceLibrary(in_dir)
inputs.Purge()
inputs.AddTypeLibrary("ncbi", DataTypeLibrary.Load(MLIB/"data_types/ncbi.yml"))
inputs.AddTypeLibrary("sequences", DataTypeLibrary.Load(MLIB/"data_types/sequences.yml"))
inputs.AddTypeLibrary("pangenome", DataTypeLibrary.Load(MLIB/"data_types/pangenome.yml"))

group = inputs.AddValue("pangenome", "e coli", "pangenome::pangenome")
inputs.AddValue("DH10b", "GCF_000019425.1", "ncbi::accession", parents={group})
inputs.AddValue("K12", "GCF_000005845.2", "ncbi::accession", parents={group})
inputs.AddItem(Path("epi300.gbk").resolve(), "sequences::gbk", parents={group})
inputs.LocalizeContents()
inputs.Save()

In [ ]:
resources = [
    DataInstanceLibrary.Load(MLIB/f"resources/{n}")
    for n in ["containers", "lib"]
]

transforms = [
    TransformInstanceLibrary.Load(MLIB/f"transforms/{n}")
    for n in ["logistics", "pangenome"]
]

task = smith.GenerateWorkflow(
    samples=inputs.AsSamples(),
    resources=resources,
    transforms=transforms,
    # targets=[inputs.GetType("sequences::gbk")]
    targets=[inputs.GetType("pangenome::heatmap")]
)

print(f'generated plan has [{len(task.plan.steps)}] steps')
display(SVG(filename=task.plan.RenderDAG("dag.svg")))

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
smith.RunWorkflow(
    task,
    config_file=smith.GetNxfConfigPresets()["local"],
)